# 05 — Fine-tuning de Modelos Transformer para Detección de Fake News en Español

Este notebook documenta el proceso de fine-tuning de cuatro configuraciones de modelos Transformer
pre-entrenados en español para la tarea de clasificación binaria de noticias (REAL / FAKE).
Se evalúan dos escenarios experimentales: **E0** (dataset original) y **E1** (dataset con
augmentación sintética), con dos arquitecturas base distintas, lo que da lugar a cuatro
experimentos independientes.

**Métricas de evaluación primaria:** F1-macro, F1-fake (clase positiva), Accuracy, Recall-fake.

**Criterio de selección del mejor checkpoint:** maximización de F1-fake en el conjunto de validación.

## Sección 0 — Entorno y Dependencias

El experimento se ejecuta en **Google Colab Pro** con acelerador de hardware **GPU NVIDIA T4**
(16 GB VRAM) y **Python 3.12**. Se instalan las versiones específicas de las bibliotecas
necesarias para garantizar reproducibilidad: `transformers >= 4.47.0` provee la clase `Trainer`
con el argumento `processing_class` introducido en esa versión; `datasets 2.20.0` provee
la API `Dataset.from_pandas`; `accelerate >= 0.29.0` es requerida por el backend de entrenamiento
distribuido; `evaluate >= 0.4.1` expone las métricas F1 y accuracy; `numpy < 2.0` mantiene
compatibilidad binaria con las versiones anteriores de scikit-learn.

El kernel se reinicia automáticamente tras la instalación para asegurar que los módulos
recién instalados sean importados correctamente en la sesión siguiente.

In [ ]:
!pip install -q "transformers>=4.47.0" "accelerate>=0.29.0" \
    "datasets==2.20.0" "evaluate>=0.4.1" "scikit-learn>=1.4.0" \
    "numpy<2.0"

import os
os.kill(os.getpid(), 9)

## Sección 1 — Autenticación y Configuración del Entorno

Se autentican dos servicios externos necesarios para el experimento:

- **HuggingFace Hub:** permite descargar los pesos de los modelos pre-entrenados (MrBERT-es y
  mRoBERTa de BSC-LT) y publicar los modelos fine-tuneados en repositorios privados al finalizar
  el entrenamiento. La autenticación se realiza mediante un token de acceso almacenado como
  secreto de Colab (`HF_TOKEN`) para evitar su exposición en el código.

- **Google Drive:** proporciona almacenamiento persistente entre sesiones de Colab. Los datasets
  procesados (E0 y E1) y los checkpoints del modelo se leen y escriben en la carpeta
  `fakenews-project` montada en `/content/drive/MyDrive/`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import transformers
import datasets
import evaluate
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"CUDA         : {torch.version.cuda}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"evaluate     : {evaluate.__version__}")

## Sección 2 — Carga de Datasets

Se cargan los seis archivos CSV que conforman los dos escenarios experimentales:

- **E0 (línea base):** splits de entrenamiento, validación y test derivados del dataset original
  FakeDeS sin modificaciones adicionales. Representa la distribución natural de clases del corpus.

- **E1 (augmentación sintética):** versión aumentada del conjunto de entrenamiento mediante
  generación de paráfrasis sintéticas con un modelo de lenguaje. Los splits de validación y test
  son idénticos a los de E0 para garantizar comparabilidad directa entre escenarios.

Ambos datasets contienen las columnas `Text` (texto de la noticia) y `Category` (etiqueta binaria:
0 = REAL, 1 = FAKE). La tabla resumen al final de la celda reporta el tamaño de cada partición.

In [ ]:
import os
import pandas as pd

BASE_DIR   = '/content/drive/MyDrive/fakenews-project'
MODELS_DIR = f"{BASE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

train_E0 = pd.read_csv(f"{BASE_DIR}/data/E0/train.csv")
val_E0   = pd.read_csv(f"{BASE_DIR}/data/E0/val.csv")
test_E0  = pd.read_csv(f"{BASE_DIR}/data/E0/test.csv")
train_E1 = pd.read_csv(f"{BASE_DIR}/data/E1/train.csv")
val_E1   = pd.read_csv(f"{BASE_DIR}/data/E1/val.csv")
test_E1  = pd.read_csv(f"{BASE_DIR}/data/E1/test.csv")

# Tabla resumen
print(f"{'Split':<12} {'E0':>6} {'E1':>6}")
print("-" * 26)
for nombre, d0, d1 in [
    ("train", train_E0, train_E1),
    ("val",   val_E0,   val_E1),
    ("test",  test_E0,  test_E1)
]:
    print(f"{nombre:<12} {len(d0):>6} {len(d1):>6}")

## Sección 3 — Parámetros del Experimento

Los hiperparámetros se fijan de forma global para todos los experimentos con el objetivo de
aislar el efecto de la arquitectura y el escenario de datos como variables independientes:

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `SEED` | 42 | Semilla estándar para reproducibilidad; inicializa generadores de PyTorch, NumPy y Python. |
| `MAX_LEN` | 512 | Longitud máxima de contexto de los modelos BERT-based; cubre el 98% de los textos del corpus. |
| `BASE_LR` | 2e-5 | Rango canónico para fine-tuning de Transformers según Devlin et al. (2019); evita catastrofic forgetting. |
| `WEIGHT_DECAY` | 0.01 | Regularización L2 estándar aplicada a todos los parámetros excepto biases y LayerNorm. |

**Modelos:** Se utilizan las versiones del Barcelona Supercomputing Center (BSC-LT) en reemplazo
de los repositorios `PlanTL-GOB-ES`, que fueron deprecados oficialmente y migrados a `BSC-LT`.
MrBERT-es emplea batch size 16 con 3 épocas; mRoBERTa emplea batch size 8 con 5 épocas,
ajustado al mayor tamaño de secuencia efectiva que procesa este modelo.

La selección del mejor checkpoint se realiza sobre la métrica **F1-fake** en validación,
ya que la clase FAKE es la de mayor interés operativo en un sistema de detección de desinformación.

In [ ]:
SEED         = 42
MAX_LEN      = 512
WEIGHT_DECAY = 0.01
BASE_LR      = 2e-5

# Modelos actualizados conforme a decisión técnica documentada
# (reemplazo de PlanTL-GOB-ES por BSC-LT por deprecación oficial)
MODELOS = {
    "mrbert-es": ("BSC-LT/MrBERT-es", 16, 3),
    "mroberta":  ("BSC-LT/mRoBERTa",   8, 5),
}

EXPERIMENTOS = [
    ("mrbert-es", "E0", train_E0, val_E0, test_E0),
    ("mrbert-es", "E1", train_E1, val_E1, test_E1),
    ("mroberta",  "E0", train_E0, val_E0, test_E0),
    ("mroberta",  "E1", train_E1, val_E1, test_E1),
]

print("Configuración del experimento:")
print(f"  Semilla          : {SEED}")
print(f"  Longitud máxima  : {MAX_LEN} tokens")
print(f"  Learning rate    : {BASE_LR}")
print(f"  Weight decay     : {WEIGHT_DECAY}")
for nombre, (model_id, batch, epochs) in MODELOS.items():
    print(f"  {nombre:<12}: {model_id} | batch={batch} | epochs={epochs}")

## Sección 4 — Función de Métricas

Se definen cuatro métricas de evaluación complementarias para caracterizar el desempeño
del clasificador en la tarea de detección de fake news:

- **F1-macro:** promedio aritmético del F1 por clase; penaliza el desbalance al ponderar
  igualmente REAL y FAKE independientemente de su frecuencia en el corpus.
- **F1-fake:** F1 de la clase positiva (FAKE); métrica principal dado el objetivo operativo
  del sistema. Alta precisión y alto recall sobre noticias falsas es la condición de diseño.
- **Accuracy:** proporción de predicciones correctas sobre el total; complementaria al F1
  en datasets con distribución moderadamente balanceada.
- **Recall-fake:** fracción de noticias falsas correctamente identificadas; relevante para
  minimizar falsos negativos (fake news que el sistema deja pasar).

La función `compute_metrics` es llamada por `Trainer` al final de cada época de validación
y sobre el conjunto de test durante la evaluación final.

In [ ]:
import numpy as np
import evaluate
from sklearn.metrics import recall_score

f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_macro    = f1_metric.compute(
        predictions=preds, references=labels, average="macro")["f1"]
    f1_fake     = f1_metric.compute(
        predictions=preds, references=labels, average=None)["f1"][1]
    acc         = acc_metric.compute(
        predictions=preds, references=labels)["accuracy"]
    recall_fake = recall_score(labels, preds, pos_label=1)
    return {
        "f1_macro"   : round(f1_macro, 4),
        "f1_fake"    : round(f1_fake, 4),
        "accuracy"   : round(acc, 4),
        "recall_fake": round(float(recall_fake), 4),
    }

## Sección 5 — Pipeline de Fine-Tuning

El pipeline itera sobre los cuatro experimentos definidos en `EXPERIMENTOS` y ejecuta para
cada uno el siguiente proceso:

1. **Tokenización:** Se instancia el tokenizador asociado al modelo y se aplica sobre los
   DataFrames de train, val y test mediante `Dataset.from_pandas` + `map`. El padding
   dinámico (`padding=False` en `tokenize` + `DataCollatorWithPadding`) es más eficiente
   en memoria que el padding estático a longitud fija.

2. **Carga del modelo:** `AutoModelForSequenceClassification` instancia la arquitectura base
   y reemplaza la cabeza de clasificación por una nueva capa lineal de 2 salidas
   (REAL / FAKE). `ignore_mismatched_sizes=True` suprime la advertencia esperada por el
   reemplazo de la capa de clasificación.

3. **Entrenamiento:** `Trainer` gestiona el ciclo de entrenamiento con evaluación por época,
   guardado del mejor checkpoint según F1-fake en validación y precisión mixta FP16
   (`fp16=True`) para acelerar el cómputo en GPU T4.

4. **Evaluación en test:** Se reportan las cuatro métricas sobre el conjunto de test usando
   el mejor checkpoint cargado automáticamente por `load_best_model_at_end=True`.

5. **Persistencia:** Modelo y tokenizador se guardan localmente en Google Drive. La memoria
   GPU se libera explícitamente entre experimentos para evitar OOM en la GPU T4.

In [ ]:
import torch
import json
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset

resultados = {}

for modelo_id, escenario, df_train, df_val, df_test in EXPERIMENTOS:
    nombre_exp             = f"{modelo_id}_{escenario}"
    model_name, batch_size, epochs = MODELOS[modelo_id]

    print(f"\n{'='*60}")
    print(f"Experimento : {nombre_exp}")
    print(f"Modelo      : {model_name}")
    print(f"Escenario   : {escenario} | batch={batch_size} | epochs={epochs}")
    print(f"{'='*60}")

    output_dir = f"{MODELS_DIR}/{nombre_exp}"
    os.makedirs(output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch["Text"],
            truncation=True,
            max_length=MAX_LEN,
            padding=False
        )

    ds_train = Dataset.from_pandas(
        df_train.rename(columns={"Category": "labels"}))
    ds_val   = Dataset.from_pandas(
        df_val.rename(columns={"Category": "labels"}))
    ds_test  = Dataset.from_pandas(
        df_test.rename(columns={"Category": "labels"}))

    ds_train = ds_train.map(tokenize, batched=True).remove_columns(["Text"])
    ds_val   = ds_val.map(tokenize,   batched=True).remove_columns(["Text"])
    ds_test  = ds_test.map(tokenize,  batched=True).remove_columns(["Text"])

    ds_train.set_format("torch")
    ds_val.set_format("torch")
    ds_test.set_format("torch")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = BASE_LR,
        weight_decay                = WEIGHT_DECAY,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1_fake",
        greater_is_better           = True,
        seed                        = SEED,
        fp16                        = True,
        report_to                   = "none",
        logging_steps               = 50,
    )

    trainer = Trainer(
        model            = model,
        args             = args,
        train_dataset    = ds_train,
        eval_dataset     = ds_val,
        processing_class = tokenizer,
        data_collator    = collator,
        compute_metrics  = compute_metrics,
    )

    trainer.train()

    test_results = trainer.evaluate(ds_test)
    print(f"\nResultados en test — {nombre_exp}:")
    print(f"  F1-macro   : {test_results['eval_f1_macro']}")
    print(f"  F1-fake    : {test_results['eval_f1_fake']}")
    print(f"  Accuracy   : {test_results['eval_accuracy']}")
    print(f"  Recall-fake: {test_results['eval_recall_fake']}")

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    resultados[nombre_exp] = {
        "modelo"     : model_name,
        "escenario"  : escenario,
        "epocas"     : epochs,
        "lr"         : BASE_LR,
        "batch_size" : batch_size,
        "f1_macro"   : test_results["eval_f1_macro"],
        "f1_fake"    : test_results["eval_f1_fake"],
        "accuracy"   : test_results["eval_accuracy"],
        "recall_fake": test_results["eval_recall_fake"],
    }

    del model, trainer
    torch.cuda.empty_cache()

## Sección 6 — Análisis de Resultados

Se presentan dos análisis comparativos sobre los resultados obtenidos en el conjunto de test:

**Tabla comparativa global:** muestra las cuatro métricas para los cuatro experimentos,
permitiendo identificar qué arquitectura y qué escenario de datos produce el mejor desempeño
en detección de fake news.

**Impacto de la augmentación sintética:** cuantifica el delta entre E1 y E0 para cada
arquitectura en las métricas F1-fake y Recall-fake. Un delta positivo indica que la
augmentación mejora la capacidad del modelo para detectar noticias falsas; un delta negativo
sugiere que las paráfrasis sintéticas introducen ruido que perjudica la generalización.

In [ ]:
print(f"\n{'='*60}")
print("REPORTE FINAL — 4 MODELOS")
print(f"{'='*60}")
print(f"{'Modelo':<20} {'Escen':>6} {'Ep':>3} {'F1-macro':>9} "
      f"{'F1-fake':>9} {'Accuracy':>9} {'Recall-fake':>12}")
print("-" * 72)
for nombre, m in resultados.items():
    print(f"{nombre:<20} {m['escenario']:>6} {m['epocas']:>3} "
          f"{m['f1_macro']:>9.4f} {m['f1_fake']:>9.4f} "
          f"{m['accuracy']:>9.4f} {m['recall_fake']:>12.4f}")

In [ ]:
print("\nImpacto de la augmentación sintética (E1 vs E0):")
print(f"{'Modelo':<12} {'ΔF1-fake':>10} {'ΔRecall-fake':>14} {'Veredicto':>12}")
print("-" * 52)
for modelo_id in ["mrbert-es", "mroberta"]:
    e0 = resultados.get(f"{modelo_id}_E0", {})
    e1 = resultados.get(f"{modelo_id}_E1", {})
    if e0 and e1:
        delta_f1     = e1['f1_fake']     - e0['f1_fake']
        delta_recall = e1['recall_fake'] - e0['recall_fake']
        veredicto    = "MEJORA" if delta_f1 > 0 else "SIN MEJORA"
        print(f"{modelo_id:<12} {delta_f1:>+10.4f} {delta_recall:>+14.4f} {veredicto:>12}")

## Sección 7 — Guardado de Resultados

Los resultados numéricos de los cuatro experimentos se serializan en formato JSON y se
persisten en Google Drive bajo la ruta `fakenews-project/resultados_finetune.json`.
Este archivo sirve como registro permanente del experimento y como fuente de datos para
análisis comparativos posteriores (e.g., tablas en el informe final, gráficos de barras).

El uso de `ensure_ascii=False` garantiza que los caracteres Unicode del español
(tildes, eñes) se almacenen de forma legible en el archivo JSON.

In [ ]:
ruta_reporte = f"{BASE_DIR}/resultados_finetune.json"
with open(ruta_reporte, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"Reporte guardado en: {ruta_reporte}")
print(json.dumps(resultados, indent=2, ensure_ascii=False))

## Sección 8 — Publicación en HuggingFace Hub

Los cuatro modelos fine-tuneados se publican como **repositorios privados** en la cuenta
de HuggingFace del usuario (`SebastianCruz10`). Esta decisión de diseño responde a los
siguientes requisitos del sistema:

- **Carga dinámica desde el backend FastAPI:** el servicio de inferencia no embebe los pesos
  del modelo en la imagen Docker, sino que los descarga en tiempo de arranque desde el Hub
  mediante `AutoModelForSequenceClassification.from_pretrained(repo_id)`. Esto mantiene la
  imagen liviana y permite actualizar el modelo sin reconstruir el contenedor.

- **Control de acceso:** los repositorios privados garantizan que solo el backend autenticado
  con el token `HF_TOKEN` pueda descargar los pesos, evitando distribución no autorizada del
  modelo entrenado.

- **Versionado de modelos:** HuggingFace Hub almacena el historial de subidas, permitiendo
  revertir a versiones anteriores del modelo en caso de regresión de desempeño.

La nomenclatura de los repositorios sigue el patrón `fakenews-{arquitectura}-{escenario}`
para facilitar la resolución programática del `repo_id` desde el backend.

In [ ]:
from huggingface_hub import HfApi

api     = HfApi()
USUARIO = "SebastianCruz10"

repos_modelos = [
    ("mrbert-es_E1", "fakenews-mrbert-es-E1"),
    ("mrbert-es_E0", "fakenews-mrbert-es-E0"),
    ("mroberta_E0",  "fakenews-mroberta-E0"),
    ("mroberta_E1",  "fakenews-mroberta-E1"),
]

for carpeta, repo_name in repos_modelos:
    repo_id = f"{USUARIO}/{repo_name}"
    print(f"Publicando {carpeta} → {repo_id}")
    api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
    api.upload_folder(
        folder_path = f"{MODELS_DIR}/{carpeta}",
        repo_id     = repo_id,
        repo_type   = "model"
    )
    print(f"OK - {repo_id}")

print("\nTodos los modelos publicados en HuggingFace.")